# How do you use pretrained models and fine-tune?
**Topics:** tf.keras.applications · Feature Extraction · Fine-Tuning · Differential LRs · HuggingFace with TF Backend

---
## 1. tf.keras.applications

### What it is
- Built-in pretrained models in Keras — trained on ImageNet
- Available models: ResNet, EfficientNet, MobileNet, VGG, InceptionV3, Xception, etc.
- Include preprocessing functions matched to each model

### Key arguments
- `weights='imagenet'` — load pretrained weights
- `include_top=False` — exclude the final classification layers
- `input_shape` — specify if different from default (224×224×3 for most models)
- `pooling='avg'` — add global average pooling as last layer when `include_top=False`

### Gotchas
- Each model has a specific preprocessing function — always use it
- `ResNet50` expects `[-1, 1]` range, `MobileNet` expects `[-1, 1]`, `EfficientNet` expects `[0, 255]`
- Use `tf.keras.applications.resnet50.preprocess_input(x)` — don't manually normalize

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load pretrained EfficientNetB0 without top
base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
    pooling='avg',
)
print(f"Base model output shape: {base_model.output_shape}")
print(f"Total params           : {base_model.count_params():,}")

# Full model with custom head
inputs = tf.keras.Input(shape=(224, 224, 3))
x      = tf.keras.applications.efficientnet.preprocess_input(inputs)
x      = base_model(x, training=False)
x      = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(5, activation='softmax')(x)
model  = tf.keras.Model(inputs, outputs)
model.summary()

# Compare available models
models_info = {
    'MobileNetV2'  : tf.keras.applications.MobileNetV2,
    'EfficientNetB0': tf.keras.applications.EfficientNetB0,
    'ResNet50'     : tf.keras.applications.ResNet50,
    'VGG16'        : tf.keras.applications.VGG16,
}

names, param_counts = [], []
for name, fn in models_info.items():
    m = fn(weights=None, include_top=False, pooling='avg')
    names.append(name)
    param_counts.append(m.count_params() / 1e6)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ECC71', '#3498DB', '#E74C3C', '#9B59B6']
bars   = ax.bar(names, param_counts, color=colors, alpha=0.85, edgecolor='white', width=0.5)
ax.set_ylabel('Parameters (millions)', fontsize=11)
ax.set_title('Pretrained Model Size Comparison (without top)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar, val in zip(bars, param_counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{val:.1f}M', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('tf_pretrained_models.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 2. Feature Extraction (Frozen Base)

### What it is
- Freeze all pretrained layers — set `base_model.trainable = False`
- Only the new head is trained
- Fastest to train, safest for small datasets

### Key intuition
- Pretrained features (edges, textures, shapes) transfer well across vision tasks
- Freezing prevents catastrophic forgetting
- Training only head: far fewer parameters → faster, less overfitting

### Gotchas
- Set `base_model.trainable = False` BEFORE `model.compile()` — compile bakes in trainable status
- If you change `trainable` after compiling, recompile the model
- Pass `training=False` to base model in `call()` — ensures BN uses running stats even during training

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

# Simulate image data (small, fast)
X_train = np.random.randn(200, 32, 32, 3).astype(np.float32)
y_train = np.random.randint(0, 5, 200)
X_val   = np.random.randn(50,  32, 32, 3).astype(np.float32)
y_val   = np.random.randint(0, 5, 50)

# Use MobileNetV2 with small input
base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, pooling='avg',
    input_shape=(32, 32, 3),
)
base.trainable = False   # freeze all pretrained layers

trainable_before = sum(1 for l in base.layers if l.trainable)
frozen_before    = sum(1 for l in base.layers if not l.trainable)
print(f"Trainable layers in base: {trainable_before}")
print(f"Frozen layers in base   : {frozen_before}")

inputs  = tf.keras.Input(shape=(32, 32, 3))
x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x       = base(x, training=False)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(5, activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs)

total     = model.count_params()
trainable = sum(l.count_params() for l in model.layers if l.trainable)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)")

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_frozen = model.fit(X_train, y_train, epochs=15, batch_size=32,
                            validation_data=(X_val, y_val), verbose=0)

# Visualize frozen vs trainable
layer_types  = [type(l).__name__ for l in model.layers]
trainable_lyr = [l.trainable for l in model.layers]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

frozen_count    = sum(1 for t in trainable_lyr if not t)
trainable_count = sum(1 for t in trainable_lyr if t)
axes[0].pie([frozen_count, trainable_count],
            labels=[f'Frozen ({frozen_count})', f'Trainable ({trainable_count})'],
            colors=['#AEB6BF', '#E74C3C'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Layer Status: Frozen vs Trainable', fontsize=12, fontweight='bold')

epochs = range(1, len(history_frozen.history['loss'])+1)
axes[1].plot(epochs, history_frozen.history['val_accuracy'], color='#2ECC71', linewidth=2.5, label='Feature extraction')
axes[1].set_title('Feature Extraction Val Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('Feature Extraction: Frozen Base + New Head', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('tf_feature_extraction.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 3. Fine-Tuning

### What it is
- After training the head, unfreeze some pretrained layers and train end-to-end
- Use a very low learning rate for pretrained layers — preserve learned features

### Two-phase strategy
1. Phase 1: freeze base, train head only (10-20 epochs)
2. Phase 2: unfreeze top layers, train with low LR (10-20 more epochs)

### Key intuition
- Phase 1 stabilizes the head — random init head won't destroy pretrained weights
- Phase 2 adapts pretrained features to your specific domain
- Fine-tuning top layers only — bottom layers have universal features (edges) that don't need updating

### Gotchas
- Recompile model after changing `trainable` attribute
- Use much lower LR for fine-tuning — 10x to 100x lower than head training
- Unfreeze gradually from top — don't unfreeze everything at once

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42); tf.random.set_seed(42)

X_train = np.random.randn(300, 32, 32, 3).astype(np.float32)
y_train = np.random.randint(0, 5, 300)
X_val   = np.random.randn(75,  32, 32, 3).astype(np.float32)
y_val   = np.random.randint(0, 5, 75)

# Build model
base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, pooling='avg', input_shape=(32,32,3))
base.trainable = False

inputs  = tf.keras.Input(shape=(32,32,3))
x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x       = base(x, training=False)
x       = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(5, activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs)

# Phase 1: train head only
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(X_train, y_train, epochs=10, batch_size=32,
               validation_data=(X_val, y_val), verbose=0)

# Phase 2: unfreeze top 30 layers of base
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

trainable_now = sum(l.count_params() for l in model.layers if l.trainable)
print(f"Phase 2 trainable params: {trainable_now:,}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # 100x lower LR
    loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
h2 = model.fit(X_train, y_train, epochs=10, batch_size=32,
               validation_data=(X_val, y_val), verbose=0)

# Combine histories
all_val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
all_val_loss = h1.history['val_loss'] + h2.history['val_loss']
all_epochs   = range(1, len(all_val_acc)+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(all_epochs, all_val_acc, color='#3498DB', linewidth=2.5)
axes[0].axvline(10.5, color='#E74C3C', linewidth=2, linestyle='--', label='Start fine-tuning')
axes[0].set_title('Validation Accuracy: Phase 1 + Phase 2', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].plot(all_epochs, all_val_loss, color='#E74C3C', linewidth=2.5)
axes[1].axvline(10.5, color='#3498DB', linewidth=2, linestyle='--', label='Start fine-tuning')
axes[1].set_title('Validation Loss: Phase 1 + Phase 2', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('Two-Phase Fine-Tuning Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tf_finetuning.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 4. HuggingFace with TensorFlow Backend

### What it is
- HuggingFace transformers supports TensorFlow via TFAutoModel classes
- Same pretrained models (BERT, GPT-2, DistilBERT) but using TF/Keras API
- Integrates with Keras training loop via `model.fit()`

### Key classes
- `TFAutoModel` — base model without task head
- `TFAutoModelForSequenceClassification` — classification head
- `TFAutoModelForTokenClassification` — NER, POS tagging
- `TFAutoModelForQuestionAnswering` — span extraction

### Gotchas
- Use `TF*` prefix for TensorFlow models — `AutoModel` gives PyTorch by default
- HuggingFace datasets return dicts — use `model.fit(dataset)` with tf.data
- Tokenizer is the same as PyTorch — framework-agnostic

### Interview Q&A

**PyTorch vs TensorFlow for HuggingFace fine-tuning?**
- PyTorch is more popular in research — most HuggingFace examples use PyTorch
- TF version works well for production with TF Serving integration
- Both support the same pretrained models and tokenizers

**What is the difference between TFAutoModel and TFAutoModelForSequenceClassification?**
- `TFAutoModel` — raw transformer outputs (hidden states)
- `TFAutoModelForSequenceClassification` — adds a classification head on top automatically

In [ ]:
# HuggingFace with TensorFlow reference patterns
# Requires: pip install transformers datasets

print("=== HuggingFace TensorFlow Patterns ===")
print()
print("1. Load TF model and tokenizer:")
print("")
print("from transformers import AutoTokenizer, TFAutoModelForSequenceClassification")
print("")
print("model_name = 'distilbert-base-uncased'")
print("tokenizer  = AutoTokenizer.from_pretrained(model_name)")
print("model      = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)")
print("")
print("2. Tokenize:")
print("")
print("def tokenize_function(examples):")
print("    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)")
print("")
print("tokenized = dataset.map(tokenize_function, batched=True)")
print("")
print("3. Convert to tf.data and train:")
print("")
print("tf_train = model.prepare_tf_dataset(")
print("    tokenized['train'], shuffle=True, batch_size=16")
print(")")
print("model.compile(")
print("    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),")
print("    metrics=['accuracy']")
print(")")
print("model.fit(tf_train, epochs=3)")
print("")
print("4. Save for TF Serving:")
print("")
print("model.save_pretrained('/tmp/tf_bert_model', saved_model=True)")
print("")

import numpy as np
import matplotlib.pyplot as plt

# Accuracy comparison: from scratch vs feature extract vs fine-tune (simulated)
np.random.seed(42)
epochs = np.arange(1, 21)

def smooth_curve(base, rate, noise=0.02):
    return [min(0.95, base + rate*e*(1-np.exp(-e/6)) + np.random.randn()*noise) for e in epochs]

from_scratch   = smooth_curve(0.50, 0.012, 0.025)
feature_extract = smooth_curve(0.68, 0.006, 0.015)
fine_tuned     = smooth_curve(0.72, 0.010, 0.012)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, from_scratch,    color='#AEB6BF', linewidth=2.5, label='Train from scratch')
ax.plot(epochs, feature_extract, color='#F39C12', linewidth=2.5, label='Feature extraction')
ax.plot(epochs, fine_tuned,      color='#2ECC71', linewidth=2.5, label='Fine-tuning')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Transfer Learning Strategies: Accuracy Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('transfer_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


### Interview Q&A

**What preprocessing function should you use for each model?**
- ResNet50: `tf.keras.applications.resnet50.preprocess_input()`
- MobileNetV2: `tf.keras.applications.mobilenet_v2.preprocess_input()`
- EfficientNet: `tf.keras.applications.efficientnet.preprocess_input()`
- VGG16: `tf.keras.applications.vgg16.preprocess_input()`
- Each has different normalization — using wrong one hurts performance significantly

**How do you unfreeze layers selectively in Keras?**
- `base_model.trainable = True` unfreezes all
- Then: `for layer in base_model.layers[:n]: layer.trainable = False` refreezes first n
- Always recompile after changing trainable status

**What is the recommended workflow for transfer learning in Keras?**
1. Load base model with `include_top=False, weights='imagenet'`
2. Set `base_model.trainable = False`
3. Build and compile model with new head
4. Train head for 10-20 epochs
5. Unfreeze top layers, recompile with 10-100x lower LR
6. Fine-tune for 10-20 more epochs

### Gotchas
- Always recompile after changing `trainable` — old compile has stale trainable status
- BatchNorm in base model: always pass `training=False` to base model call during fine-tuning phase
- Learning rate for fine-tuning should be 10-100x lower than head training LR